# Transfer Learning
- reduces the need for large training datasets
- cuts down training time and compute costs
- build on top of existing models that have been trained on large datasets
- achieves better performance with limited data
- improves generalization to new, unseen data

In [ ]:
import torch

In [ ]:
# load pre-trained model
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

# freeze initial layers
for param in model.parameters():
    param.requires_grad = False

# unfreeze deeper layers
for param in model.model[-2].parameters():
    param.requires_grad = True

# fine-tuning training loop

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    for images, targets in loader:
        optimizer.zero_grad()
        predictions = model(images)

        # calc iou
        iou = calculate_iou(predictions['boxes'], targets['boxes'])
        # calc mAP
        map_score = calc_map(predictions['boxes'], targets['boxes'])
        print(f'IoU: {iou}, MAP: {map_score}')

        loss = compute_loss(predictions, targets)
        loss.backward()
        optimizer.step()
        print(f'Epoch {epoch}, Loss: {loss.item()}')

# Real-Time Detection Workflow

- use a pre-trained model (e.g., YOLOv5) for object detection
- capture video frames from a webcam or video file
- perform inference on each frame using the pre-trained model
- draw bounding boxes and labels on detected objects

In [ ]:
import torch
import cv2

# load pre-trained model
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

cap = cv2.VideoCapture(0) # capture video from webcam

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: # if not source - exit from loop
        break

    # convert frame to RGB
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # perform inference
    results = model(img)

    for x1, y1, x2, y2, conf, cls in results.xyxy[0]:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f'{int(cls)}: {int(conf)} %',
                    (x1, y1), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # render results on the frame
    results.render()

    # convert back to BGR for OpenCV
    output_frame = cv2.cvtColor(results.imgs[0], cv2.COLOR_RGB2BGR)

    # display the output frame
    cv2.imshow('Real-Time Detection', output_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()